In [ ]:
# Cell 1: Setup
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_tracking_uri("http://localhost:5000")
client = MlflowClient()

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Connected to MLflow")

In [ ]:
# Cell 2: List all experiments
experiments = client.search_experiments()

print("Available Experiments:")
print("=" * 60)
for exp in experiments:
    print(f"Name: {exp.name}")
    print(f"ID: {exp.experiment_id}")
    print(f"Lifecycle: {exp.lifecycle_stage}")
    print("-" * 60)

In [ ]:
# Cell 3: Compare runs from advanced experiment
experiment_name = "fraud-detection-advanced"
experiment = client.get_experiment_by_name(experiment_name)

if experiment:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.test_auc_roc DESC"]
    )
    
    # Create comparison DataFrame
    comparison_data = []
    for run in runs:
        comparison_data.append({
            'run_id': run.info.run_id[:8],
            'model_type': run.data.params.get('model_type', 'unknown'),
            'sampling': run.data.params.get('sampling_strategy', 'none'),
            'val_auc': run.data.metrics.get('val_auc_roc', 0),
            'test_auc': run.data.metrics.get('test_auc_roc', 0),
            'precision': run.data.metrics.get('test_precision', 0),
            'recall': run.data.metrics.get('test_recall', 0),
            'f1_score': run.data.metrics.get('test_f1_score', 0),
            'training_time': run.data.metrics.get('training_time_seconds', 0)
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(f"\n{len(df_comparison)} runs found in '{experiment_name}'\n")
    df_comparison
else:
    print(f"Experiment '{experiment_name}' not found!")

In [ ]:
# Cell 4: Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['test_auc', 'precision', 'recall', 'f1_score']
titles = ['Test AUC-ROC', 'Precision', 'Recall', 'F1-Score']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    
    # Create labels combining model type and sampling
    df_comparison['label'] = df_comparison['model_type'] + '\n' + df_comparison['sampling']
    
    # Plot
    bars = ax.bar(range(len(df_comparison)), df_comparison[metric], color='steelblue', alpha=0.7)
    ax.set_xticks(range(len(df_comparison)))
    ax.set_xticklabels(df_comparison['label'], rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, df_comparison[metric])):
        ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Training time comparison
plt.figure(figsize=(12, 6))
df_comparison['label'] = df_comparison['model_type'] + ' - ' + df_comparison['sampling']
plt.barh(df_comparison['label'], df_comparison['training_time'], color='coral', alpha=0.7)
plt.xlabel('Training Time (seconds)', fontsize=12)
plt.title('Training Time Comparison', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for i, (label, time) in enumerate(zip(df_comparison['label'], df_comparison['training_time'])):
    plt.text(time, i, f' {time:.1f}s', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Precision-Recall tradeoff
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df_comparison['recall'], df_comparison['precision'], 
                     s=df_comparison['test_auc']*500, alpha=0.6, c=range(len(df_comparison)),
                     cmap='viridis')

for i, row in df_comparison.iterrows():
    label = f"{row['model_type'][:4]}-{row['sampling'][:4]}"
    plt.annotate(label, (row['recall'], row['precision']), 
                fontsize=9, alpha=0.8)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Tradeoff (bubble size = AUC)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.colorbar(scatter, label='Model Index')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Load and test best model
best_run_id = df_comparison.iloc[0]['run_id']
print(f"Loading best model (run_id: {best_run_id}...)")

# Get full run ID
full_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.runName != 'None'",
    order_by=["metrics.test_auc_roc DESC"],
    max_results=1
)

if full_runs:
    best_run_full_id = full_runs[0].info.run_id
    model_uri = f"runs:/{best_run_full_id}/model"
    
    # Load model
    loaded_model = mlflow.sklearn.load_model(model_uri)
    
    print(f"✅ Model loaded successfully!")
    print(f"   Type: {type(loaded_model)}")
    print(f"   Run ID: {best_run_full_id}")

In [ ]:
# Cell 8: Test prediction
import numpy as np

# Load test data
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Make predictions on a sample
sample_size = 10
sample_indices = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test.iloc[sample_indices]
y_sample = y_test[sample_indices]

# Predict
predictions_proba = loaded_model.predict_proba(X_sample)[:, 1]
predictions = loaded_model.predict(X_sample)

# Display results
results_df = pd.DataFrame({
    'Actual': y_sample,
    'Predicted': predictions,
    'Fraud_Probability': predictions_proba
})

print("\nSample Predictions:")
print("=" * 60)
print(results_df.to_string(index=False))
print("\n✅ Model is working correctly!")

In [ ]:
# Cell 9: Check registered models
registered_models = client.search_registered_models()

print("Registered Models:")
print("=" * 80)
for rm in registered_models:
    print(f"\nModel: {rm.name}")
    print(f"Description: {rm.description}")
    
    # Get latest versions
    for version in rm.latest_versions:
        print(f"  Version {version.version}: {version.current_stage}")
        print(f"  Created: {version.creation_timestamp}")
        print(f"  Source: {version.source}")